In [1]:
import pandas as pd
import numpy as np


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,mieszkaniowa wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,mieszkaniowa wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,mieszkaniowa wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,centrum medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1236,205167460,NaN,NaN,NaN,techniczna,NaN,NaN,1,<NA>,1980,współczesność,2,NaN,NaN,google.com/maps,2,2026-09-30,1
1237,126180359,Władysława IV,61,NaN,"laboratorium medyczne,piekarnia,kiosk",NaN,mieszkaniowa wielorodzinna,7,<NA>,1940,modernizm,3,NaN,NaN,google.com/maps,2,2026-09-30,3
1238,126173503,Świętojańska,122,NaN,"apteka,handel zdrowie",NaN,biurowa,6,<NA>,1930,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-30,2


In [3]:
data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long_categories.head()

,addr_street,function_clean,category
id,,,
219,Stefana Żeromskiego,administracja,organizacje publiczne
258,3 Maja,administracja,organizacje publiczne
374,Starowiejska,administracja,organizacje publiczne
495,10 Lutego,administracja,organizacje publiczne
86,Jana z Kolna,administracja,organizacje publiczne


1.

In [4]:
bud = data_gdynia.groupby(["addr_street", "arch_style"]).size()
bud.head()

addr_street  arch_style   
10 Lutego    klasycyzm        14
             modernizm        12
             współczesność    12
3 Maja       klasycyzm         2
             modernizm        11
dtype: int64

In [5]:
bud.loc["Świętojańska"]

arch_style
klasycyzm         9
modernizm        90
współczesność     7
dtype: int64

In [6]:
bud.xs("klasycyzm", level="arch_style")

addr_street
10 Lutego              14
3 Maja                  2
Antoniego Abrahama      4
Armii Krajowej          1
Jerzego Waszyngtona     2
Plac Grunwaldzki        1
Plac Kaszubski          2
Starowiejska            5
Wójta Radtkego          2
Władysława IV           1
Zgoda                   6
Świętojańska            9
dtype: int64

In [7]:
assert bud.index.nlevels == 2
assert bud.sum() == data_gdynia["addr_street"].notna().sum()

2.

In [8]:
szer = bud.unstack()
szer0 = bud.unstack(fill_value=0)
szer.dtypes

arch_style
klasycyzm         float64
modernizm         float64
postmodernizm     float64
tradycjonalizm    float64
współczesność     float64
dtype: object

In [9]:
assert szer.isna().sum().sum() == 126
assert (szer0.sum(axis=1) == data_gdynia.groupby("addr_street").size()).all()

3.

In [10]:
pd.testing.assert_series_equal(szer0.stack().loc[szer0.stack() > 0], bud)

In [11]:
szer.stack()

addr_street                         arch_style    
10 Lutego                           klasycyzm         14.0
                                    modernizm         12.0
                                    postmodernizm      NaN
                                    tradycjonalizm     NaN
                                    współczesność     12.0
                                                      ... 
Żołnierzy I Armii Wojska Polskiego  klasycyzm          NaN
                                    modernizm          2.0
                                    postmodernizm      NaN
                                    tradycjonalizm     NaN
                                    współczesność      1.0
Length: 220, dtype: float64

unstack stack witgout filling Nan values, leaves Nan values in the series

4.

In [12]:
arch_style_year = data_gdynia.groupby(["arch_style", "build_year_est"]).size()
arch_style_year

arch_style      build_year_est
klasycyzm       1910                1
                1920               21
                1930               27
modernizm       1920                2
                1930              191
                1940               79
                1950              187
                1960                4
                1970                3
postmodernizm   1980                3
                2000                1
tradycjonalizm  1900                1
                1930                5
                1940                1
współczesność   1970                1
                1980              342
                1990                8
                2000               65
                2010                3
                2020               37
dtype: int64

In [13]:
arch_style_year.unstack()

build_year_est,1900,1910,1920,1930,1940,1950,1960,1970,1980,1990,2000,2010,2020
arch_style,,,,,,,,,,,,,
klasycyzm,NaN,1.0,21.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modernizm,NaN,NaN,2.0,191.0,79.0,187.0,4.0,3.0,NaN,NaN,NaN,NaN,NaN
postmodernizm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,1.0,NaN,NaN
tradycjonalizm,1.0,NaN,NaN,5.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
współczesność,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,342.0,8.0,65.0,3.0,37.0


In [14]:
arch_style_year.unstack("arch_style")

arch_style,klasycyzm,modernizm,postmodernizm,tradycjonalizm,współczesność
build_year_est,,,,,
1900,NaN,NaN,NaN,1.0,NaN
1910,1.0,NaN,NaN,NaN,NaN
1920,21.0,2.0,NaN,NaN,NaN
1930,27.0,191.0,NaN,5.0,NaN
1940,NaN,79.0,NaN,1.0,NaN
1950,NaN,187.0,NaN,NaN,NaN
1960,NaN,4.0,NaN,NaN,NaN
1970,NaN,3.0,NaN,NaN,1.0
1980,NaN,NaN,3.0,NaN,342.0


In [15]:
arch_style_year.unstack(level=0)

arch_style,klasycyzm,modernizm,postmodernizm,tradycjonalizm,współczesność
build_year_est,,,,,
1900,NaN,NaN,NaN,1.0,NaN
1910,1.0,NaN,NaN,NaN,NaN
1920,21.0,2.0,NaN,NaN,NaN
1930,27.0,191.0,NaN,5.0,NaN
1940,NaN,79.0,NaN,1.0,NaN
1950,NaN,187.0,NaN,NaN,NaN
1960,NaN,4.0,NaN,NaN,NaN
1970,NaN,3.0,NaN,NaN,1.0
1980,NaN,NaN,3.0,NaN,342.0


In [16]:
pd.testing.assert_frame_equal(
    arch_style_year.unstack().T, arch_style_year.unstack("arch_style")
)
pd.testing.assert_frame_equal(
    arch_style_year.unstack(level=0), arch_style_year.unstack("arch_style")
)

5.

In [17]:
t = bud.reset_index(name="num_build")
t.pivot(index="addr_street", columns="arch_style", values="num_build")

arch_style,klasycyzm,modernizm,postmodernizm,tradycjonalizm,współczesność
addr_street,,,,,
10 Lutego,14.0,12.0,NaN,NaN,12.0
3 Maja,2.0,11.0,NaN,NaN,8.0
Aleja Jana Pawła II,NaN,4.0,NaN,NaN,11.0
Aleja Marszałka Józefa Piłsudskiego,NaN,2.0,NaN,NaN,NaN
Antoniego Abrahama,4.0,50.0,NaN,NaN,16.0
Antoniego Hryniewickiego,NaN,10.0,NaN,NaN,13.0
Arkadiusza Rybickiego,NaN,NaN,NaN,NaN,9.0
Armii Krajowej,1.0,19.0,NaN,NaN,5.0
Dworcowa,NaN,2.0,NaN,NaN,2.0


In [18]:
try:
    data_gdynia.pivot(
        index="addr_street", columns="arch_style", values="number_of_storeys"
    )
except ValueError as e:
    print(e)

Index contains duplicate entries, cannot reshape


In [19]:
pd.testing.assert_frame_equal(t.pivot(index="addr_street", columns="arch_style", values="num_build"), szer)

6.

In [20]:
def only_big_streets(df, min_build):
    return df.loc[df.groupby("addr_street").transform("size") >= min_build]

big = data_gdynia.pipe(only_big_streets, min_build = 20)


In [21]:
pd.testing.assert_frame_equal(big.groupby(['addr_street','arch_style'])['number_of_storeys'].median().unstack(), big.pivot_table(index = 'addr_street', columns = 'arch_style', values = 'number_of_storeys', aggfunc = 'median'))

In [22]:
big.pivot_table(index = 'addr_street', columns = 'arch_style', values = 'number_of_storeys')

arch_style,klasycyzm,modernizm,postmodernizm,tradycjonalizm,współczesność
addr_street,,,,,
10 Lutego,3.857143,6.083333,<NA>,<NA>,2.416667
3 Maja,2.0,6.090909,<NA>,<NA>,4.25
Antoniego Abrahama,5.0,5.54,<NA>,<NA>,2.25
Antoniego Hryniewickiego,<NA>,2.8,<NA>,<NA>,4.769231
Armii Krajowej,6.0,5.421053,<NA>,<NA>,5.2
Jana z Kolna,<NA>,3.3,<NA>,<NA>,2.571429
Jerzego Waszyngtona,5.5,4.0,<NA>,2.5,6.153846
Starowiejska,4.0,5.638889,<NA>,2.0,2.454545
Stefana Żeromskiego,<NA>,4.210526,<NA>,4.0,5.666667


pivot table counts mean by default

7.

In [23]:
data_gdynia.pivot_table(index = 'arch_style', columns = 'condition', values = 'osm_id', aggfunc = 'count', fill_value= 0, margins = True, margins_name= 'Sum')


condition,1,2,3,Sum
arch_style,,,,
klasycyzm,0,32,17,49
modernizm,18,338,110,466
postmodernizm,0,2,2,4
tradycjonalizm,0,4,3,7
współczesność,13,354,89,456
Sum,31,730,221,982


8.

In [24]:
tab = data_gdynia.pivot_table(index = 'arch_style', columns = 'condition', values = 'osm_id', aggfunc = 'count', fill_value= 0, margins = False)
tab

condition,1,2,3
arch_style,,,
klasycyzm,0,32,17
modernizm,18,338,110
postmodernizm,0,2,2
tradycjonalizm,0,4,3
współczesność,13,354,89


In [25]:
style_condition = tab.div(tab.sum(axis =1 ), axis = 0)
style_condition

condition,1,2,3
arch_style,,,
klasycyzm,0.000000,0.653061,0.346939
modernizm,0.038627,0.725322,0.236052
postmodernizm,0.000000,0.500000,0.500000
tradycjonalizm,0.000000,0.571429,0.428571
współczesność,0.028509,0.776316,0.195175


In [26]:
assert np.isclose(style_condition.sum(axis = 1), 1).all()

9.

In [27]:
m = szer0.reset_index().melt(id_vars = 'addr_street', var_name = 'arch_style', value_name = 'num_build') 
m

,addr_street,arch_style,num_build
0,10 Lutego,klasycyzm,14
1,3 Maja,klasycyzm,2
2,Aleja Jana Pawła II,klasycyzm,0
3,Aleja Marszałka Józefa Piłsudskiego,klasycyzm,0
4,Antoniego Abrahama,klasycyzm,4
...,...,...,...
215,Śledziowa,współczesność,0
216,Świętego Piotra,współczesność,4
217,Świętego Wojciecha,współczesność,1
218,Świętojańska,współczesność,7


In [28]:
pd.testing.assert_frame_equal(t,m.loc[m['num_build'] != 0].sort_values(['addr_street', 'arch_style'], ascending= True).reset_index(drop= True) )

10.

In [29]:
lok = data_gdynia_long_categories.join(data_gdynia[['arch_style']], validate= 'many_to_one')
p10 = lok.pivot_table(index= 'category', columns = 'arch_style', aggfunc='size', fill_value=0)
p10 = p10.reset_index().melt(id_vars = 'category', var_name = 'arch_style', value_name = 'num_build')
p10 = p10.loc[p10['num_build']>0].set_index(['category', 'arch_style']).sort_index()

In [30]:
pd.testing.assert_series_equal(p10['num_build'], lok.groupby(['category','arch_style']).size().rename('num_build'))